# MichiganCast Demo 01 — Data Foundation（数据基础能力）

本 notebook 聚焦展示三件事：

1. 数据契约与质量校验（可定位问题）。
2. 可复现清洗流水线（替代手工 notebook 步骤）。
3. 数据版本追踪（可回溯来源与生成命令）。

## 0. 本章目标与结论

**目标：** 演示你在数据工程方向的工程能力，而不仅是建模。

**结论模板（运行后补充）：**
- 原始数据有哪些结构与质量问题。
- 清洗后数据在契约校验下是否通过。
- 版本 manifest 能否完整回溯该数据集来源。

## 1. 输入与输出（路径与产物）

| 类型 | 路径 |
|---|---|
| 原始表格输入 | `data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv` |
| 清洗后输出 | `data/interim/traverse_city_daytime_clean_v1.csv` |
| 契约报告 | `artifacts/reports/data_contract_report.json` |
| 质量报告 | `artifacts/data_quality_report.json` |
| 清洗摘要 | `artifacts/reports/data_cleaning_summary.json` |
| 数据版本清单 | `configs/data/versions/*.json` |

In [ ]:
from pathlib import Path
import subprocess
import json

RAW_CSV = 'data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv'
CLEAN_CSV = 'data/interim/traverse_city_daytime_clean_v1.csv'
IMAGE_DIR = 'data/processed/images/lake_michigan_64_png'

print('RAW exists:', Path(RAW_CSV).exists())
print('IMAGE dir exists:', Path(IMAGE_DIR).exists())
print('CLEAN exists:', Path(CLEAN_CSV).exists())

## 2. 方法与实现（调用 `src` 模块）

约定：默认只打印命令，不直接执行；把 `DEMO_EXECUTE = True` 后再正式跑。

In [ ]:
DEMO_EXECUTE = False

def run_or_print(cmd: str):
    print('\n$ ' + cmd)
    if not DEMO_EXECUTE:
        print('[skip] DEMO_EXECUTE=False, only showing command.')
        return None
    completed = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
    return completed.returncode

### 2.1 数据契约检查（T10）

**目的：** 在清洗前定位 schema/数值范围问题。

In [ ]:
cmd_contract = (
    'scripts/run_in_pytorch_env.sh python -m src.data.contracts '
    '--input data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv '
    '--output artifacts/reports/data_contract_report.json'
)
run_or_print(cmd_contract)

### 2.2 数据质量检查（T11）

**目的：** 检查缺失码、时间连续性、图像库存和图像-表格对齐。

In [ ]:
cmd_validate = (
    'scripts/run_in_pytorch_env.sh python -m src.data.validate '
    '--input data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv '
    '--image-dir data/processed/images/lake_michigan_64_png '
    '--output artifacts/data_quality_report.json'
)
run_or_print(cmd_validate)

### 2.3 清洗流水线（T12）

**目的：** 一条命令生成干净数据，保证可复现。

In [ ]:
cmd_clean = (
    'scripts/run_in_pytorch_env.sh python -m src.data.clean '
    '--input data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv '
    '--output data/interim/traverse_city_daytime_clean_v1.csv '
    '--summary artifacts/reports/data_cleaning_summary.json'
)
run_or_print(cmd_clean)

### 2.4 数据版本追踪（T31）

**目的：** 记录数据哈希、来源与构建命令，形成可审计 lineage。

In [ ]:
cmd_version = (
    'scripts/run_in_pytorch_env.sh python -m src.data.versioning '
    '--dataset-id traverse_city_clean_v1 '
    '--layer interim '
    '--target-path data/interim/traverse_city_daytime_clean_v1.csv '
    '--source-path data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv '
    '--build-command "python -m src.data.clean --input ... --output ..." '
    '--notes "demo notebook snapshot"'
)
run_or_print(cmd_version)

## 3. 结果解读（模板）

运行完成后，建议在这里填三段简短解读：

1. **契约层结论：** 哪些列在原始数据中违规最多，是否会影响训练。
2. **清洗层结论：** 删除/修复了多少行，清洗后契约是否通过。
3. **版本层结论：** manifest 是否能完整回溯数据来源与构建命令。

In [ ]:
candidate_reports = [
    'artifacts/reports/data_contract_report.json',
    'artifacts/data_quality_report.json',
    'artifacts/reports/data_cleaning_summary.json',
    'configs/data/versions/traverse_city_clean_v1.json',
]

for rp in candidate_reports:
    p = Path(rp)
    if not p.exists():
        print(f'{rp}: NOT FOUND')
        continue
    print(f'{rp}: FOUND, size={p.stat().st_size} bytes')

## 4. 工程反思与下一步

- 反思 1：当前最脆弱的数据步骤是什么？（例如图像-表格对齐）
- 反思 2：还需要补哪些自动化测试来防回归？
- 下一步：衔接 `02_analysis_and_baselines.ipynb` 进入分析与模型基线阶段。

## Appendix. 复现实验命令

可直接复制到终端：

```bash
scripts/run_in_pytorch_env.sh python -m src.data.contracts --input data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv
scripts/run_in_pytorch_env.sh python -m src.data.validate --input data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv --image-dir data/processed/images/lake_michigan_64_png
scripts/run_in_pytorch_env.sh python -m src.data.clean --input data/processed/tabular/traverse_city_daytime_meteo_preprocessed.csv --output data/interim/traverse_city_daytime_clean_v1.csv
scripts/run_in_pytorch_env.sh python -m src.data.versioning --dataset-id traverse_city_clean_v1 --layer interim --target-path data/interim/traverse_city_daytime_clean_v1.csv
```